# 05 - GraphRAG Ablation + Multi-objective HPO  (D3 Task 5 - Owner: Yehia Noureldin)

Runs **Yehia's `graphrag_hpo.py` runner** on the live pipeline and produces the three graded
T5 artifacts: the **ablation table**, the **NSGA-II Pareto plot**, and the **recommended config**.
Built against the frozen contracts only - `GraphRAGExecutor.answer(...)` and
`evaluate_answers(answer_fn, gold)` - so whatever Tasks 1/2/3 ship underneath is invisible here.

**This notebook also closes T4's headline number:** every ablation cell calls the real RAGAS-via-Groq
`evaluate_answers`, so the production-config row *is* the measured faithfulness / answer-relevance for
Task 4. The consolidated `reports/D3/d3_ablation.md` written by the last cell records both.

## Prerequisites (this will NOT run without them)
- **Live D2 stack up:** Qdrant + Neo4j (e.g. `make up`) - the executor retrieves against them.
- **Ollama** serving `qwen2.5:3b-instruct` (the zero-shot answerer).
- **`GROQ_API_KEY`** in `.env` - RAGAS judge = Groq `llama-3.3-70b-versatile`.
- **ragas deps resolved.** If you hit the ragas/langchain_community clash, install a known-good set first:
  ```
  pip install -U "ragas>=0.2,<0.3" "langchain-core>=0.3,<0.4" "langchain-community>=0.3,<0.4" \
    "langchain-groq>=0.2,<0.3" "langchain-huggingface>=0.1" "datasets>=2.20"
  ```

> **Free-tier budget:** Groq free tier is ~30 rpm / ~1000 req/day. The ablation grid (6 cells x 40 gold rows)
> plus a small NSGA-II search fits under the daily cap *only* because the search runs on a gold **subset**
> (see Cell 6). Don't bump `N_TRIALS` / `SEARCH_N` blindly or you'll exhaust the daily quota.

## 1 - Environment
Load `.env`, confirm the Groq judge key, pick the answerer (zero-shot for the D3 ablation).

In [1]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, 'src')  # so `import csai415...` works from repo root

# load .env (no hard dependency on python-dotenv)
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            kk, vv = line.split('=', 1)
            os.environ.setdefault(kk.strip(), vv.strip().strip('"').strip("'"))

# D3 ablation uses the ZERO-SHOT answerer; D4 flips this to the tuned model (see last cell)
os.environ.setdefault('CSAI415_ANSWERER', 'qwen2.5:3b-instruct')
assert os.getenv('GROQ_API_KEY'), 'GROQ_API_KEY missing - RAGAS judge needs it (set it in .env)'

from csai415.answer import current_backend
print('answerer backend :', current_backend())
print('RAGAS judge      : Groq llama-3.3-70b-versatile')

answerer backend : ollama:qwen2.5:3b-instruct  [cite_mode=numbered]
RAGAS judge      : Groq llama-3.3-70b-versatile


## 2 - Load the gold eval set
40 hand-curated arXiv cs.CL rows (Ahmad, Task 4). The leakage assertion fires inside `evaluate_answers`.

In [2]:
import os
# Move up one directory to the project root
os.chdir('..') 
print("New working directory:", os.getcwd())

New working directory: c:\Users\yehia\special-topics


In [3]:
gold_path = Path('data/gold/qa_answers.jsonl')
gold = [json.loads(l) for l in gold_path.read_text(encoding='utf-8').splitlines() if l.strip()][:10]
print(f'loaded {len(gold)} gold rows from {gold_path}')

loaded 10 gold rows from data\gold\qa_answers.jsonl


In [4]:
import importlib.metadata

print("Scanning for corrupted packages...")
for dist in importlib.metadata.distributions():
    try:
        _ = dist.metadata['Name']
    except KeyError:
        print(f"🚨 FOUND BROKEN PACKAGE: {dist._path}")

Scanning for corrupted packages...


## 3 - Build the live executor
`get_default_executor()` wires the single-collection Qdrant + Neo4j stack (Task 7).

In [5]:
from csai415.graphrag import get_default_executor
from csai415.eval import evaluate_answers
executor = get_default_executor()
print('executor ready:', type(executor).__name__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

executor ready: GraphRAGExecutor


## 4 - Part 1: ablation grid (3 modes x rerank on/off)
Full factorial - 6 cells, each run once on the **full** gold set so interaction effects are visible
(does rerank help `hybrid` as much as `vector`?). Saves the table to `reports/D3/`.

In [6]:
from csai415.graphrag_hpo import run_ablation_grid, ablation_rows_to_markdown

rows = run_ablation_grid(executor=executor, evaluate_answers=evaluate_answers, gold=gold, k=5)
table_md = ablation_rows_to_markdown(rows)
print(table_md)

out = Path('reports/D3'); out.mkdir(parents=True, exist_ok=True)
(out / 'ablation_table.md').write_text('# T5 - GraphRAG Ablation Grid (real run)\n\n' + table_md + '\n', encoding='utf-8')
import csv
with (out / 'ablation_grid.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['mode','rerank','faithfulness','answer_relevance','recall5','p95_latency_ms','n'])
    for r in rows:
        w.writerow([r.mode, r.rerank, r.faithfulness, r.answer_relevance, r.recall5, r.p95_latency_ms, r.n])
print('\nsaved reports/D3/ablation_table.md + ablation_grid.csv')

[ablation] mode=vector rerank=False ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

[ablation] mode=vector rerank=True ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

[ablation] mode=hybrid rerank=True ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Exception raised in Job[12]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvp45ewyecxrgzw03sg54vg8` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99635, Requested 454. Please try again in 1m16.896s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[1]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvp45ewyecxrgzw03sg54vg8` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99995, Requested 425. Please try again in 6m2.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[15]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit r

| mode | rerank | faithfulness | answer_relevance | recall@5 | p95_latency_ms | n |
|---|---|---|---|---|---|---|
| vector | False | 0.966 | 0.905 | 0.400 | 5106 | 10 |
| vector | True | 0.750 | 0.874 | 0.600 | 18105 | 10 |
| graph | False | nan | 0.936 | 0.500 | 4221 | 10 |
| graph | True | nan | nan | 0.600 | 13912 | 10 |
| hybrid | False | nan | nan | 0.500 | 4314 | 10 |
| hybrid | True | nan | nan | 0.600 | 12961 | 10 |

saved reports/D3/ablation_table.md + ablation_grid.csv


## 5 - Pick the winning mode (defend it)
Winner = highest faithfulness; ties broken by lower p95 latency. NSGA-II then tunes knobs *within* this
mode (the grid already settled mode, so the search shouldn't re-litigate it).

In [7]:
best = max(rows, key=lambda r: (round(r.faithfulness, 4), -r.p95_latency_ms))
WINNING_MODE = best.mode
prod_row = max((r for r in rows if r.mode == WINNING_MODE), key=lambda r: r.faithfulness)
print(f'WINNING_MODE = {WINNING_MODE!r}  (faithfulness {best.faithfulness:.3f}, rerank={best.rerank})')
print(f'production row -> faithfulness {prod_row.faithfulness:.3f} | answer_relevance {prod_row.answer_relevance:.3f}'
      f' | recall@5 {prod_row.recall5:.3f} | p95 {prod_row.p95_latency_ms:.0f} ms')
# This production row IS Task 4's headline number:
T4_OK = prod_row.faithfulness >= 0.8 and prod_row.answer_relevance >= 0.8
print(f'T4 targets (>=0.8 / >=0.8): {"MET" if T4_OK else "NOT MET - investigate"}')

WINNING_MODE = 'vector'  (faithfulness 0.966, rerank=False)
production row -> faithfulness 0.966 | answer_relevance 0.905 | recall@5 0.400 | p95 5106 ms
T4 targets (>=0.8 / >=0.8): MET


## 6 - Part 2: NSGA-II Pareto search (faithfulness up, latency down)
Searches `(rerank, candidate_k, rerank_top_n, k)` within the winning mode. **Runs on a gold subset** to
stay inside the Groq daily cap; the chosen knee config is then validated on the full gold in Cell 8.
Two objectives, both directions declared - no pre-weighted scalar.

In [9]:
from csai415.graphrag_hpo import run_nsga2_search, pareto_to_markdown
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEARCH_N  = 5   # gold rows per trial (subset -> affordable under the free-tier daily cap)
N_TRIALS  = 1   # NSGA-II evaluations
POP_SIZE  = 6
search_gold = gold[:SEARCH_N]
print(f'searching on {len(search_gold)}/{len(gold)} gold rows, {N_TRIALS} trials...')

result = run_nsga2_search(executor=executor, evaluate_answers=evaluate_answers, gold=search_gold,
                          fixed_mode=WINNING_MODE, n_trials=N_TRIALS, population_size=POP_SIZE, seed=42)
pareto_md = pareto_to_markdown(result)
print('\n=== Pareto front (knee = star) ===')
print(pareto_md)

searching on 5/10 gold rows, 1 trials...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Exception raised in Job[6]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvp45ewyecxrgzw03sg54vg8` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99861, Requested 544. Please try again in 5m49.919999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[5]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvp45ewyecxrgzw03sg54vg8` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99832, Requested 430. Please try again in 3m46.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[4]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate l


=== Pareto front (knee = star) ===
| trial | rerank | candidate_k | rerank_top_n | k | faithfulness | p95_latency_ms | knee? |
|---|---|---|---|---|---|---|---|


## 7 - Pareto plot
All trials (grey) + Pareto front (blue) + knee point (star). Saved to `reports/D3/pareto_front.png`.

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

done = [t for t in result.study.trials if t.values is not None]
fx = [t.values[0] for t in done]; ly = [t.values[1] for t in done]
pf = sorted(result.best_trials, key=lambda t: t.values[1])
pfx = [t.values[0] for t in pf]; pfy = [t.values[1] for t in pf]

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.scatter(ly, fx, c='lightgrey', label='all trials', zorder=1)
ax.plot(pfy, pfx, '-o', c='tab:blue', label='Pareto front', zorder=2)
if result.knee_point is not None:
    k = result.knee_point
    ax.scatter([k.values[1]], [k.values[0]], marker='*', s=320, c='tab:red',
               edgecolor='black', zorder=3, label=f'knee (trial {k.number})')
ax.set_xlabel('p95 latency (ms)  -  lower better'); ax.set_ylabel('faithfulness  -  higher better')
ax.set_title(f'NSGA-II Pareto front - mode={WINNING_MODE}'); ax.legend(); fig.tight_layout()
fig.savefig('reports/D3/pareto_front.png', dpi=130)
print('saved reports/D3/pareto_front.png'); plt.show()

saved reports/D3/pareto_front.png


C:\Users\yehia\AppData\Local\Temp\ipykernel_8172\877591346.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print('saved reports/D3/pareto_front.png'); plt.show()


## 8 - Recommended config + consolidated report
Validate the knee config on the **full** gold, then write `reports/D3/d3_ablation.md` (ablation table +
winning mode + Pareto table + recommended config + T4 headline). This file is the graded T5 deliverable.

In [11]:
knee = result.knee_point
rec = {'mode': WINNING_MODE, **(knee.params if knee else {})}
print('recommended (knee) config:', rec)

# validate the knee config on the FULL gold set
val = None
if knee is not None:
    def _ans(q):
        return executor.answer(q, k=knee.params['k'], mode=WINNING_MODE, rerank=knee.params['rerank'])
    val = evaluate_answers(answer_fn=_ans, gold=gold)
    print('knee config on FULL gold:', {kk: round(vv, 3) if isinstance(vv, float) else vv for kk, vv in val.items()})

report = ['# Task 5 - GraphRAG Ablation + Multi-objective HPO (D3)\n',
          '**Owner:** Yehia Noureldin  ', '**Runner:** `src/csai415/graphrag_hpo.py`  ',
          f'**Answerer:** `{os.getenv("CSAI415_ANSWERER")}` (zero-shot)  ',
          '**Judge:** RAGAS + Groq `llama-3.3-70b-versatile`\n',
          '## 1. Ablation grid (3 modes x rerank)\n', table_md, '',
          f'**Winning mode:** `{WINNING_MODE}` - highest faithfulness, latency tie-break.\n',
          '## 2. NSGA-II Pareto front (faithfulness up / latency down)\n', pareto_md, '',
          '![Pareto front](pareto_front.png)\n',
          '## 3. Recommended config (knee point)\n', f'```\n{rec}\n```\n']
if val is not None:
    report.append(f'Knee config on full gold: faithfulness **{val["faithfulness"]:.3f}**, '
                  f'answer_relevance **{val["answer_relevance"]:.3f}**, recall@5 {val["recall5"]:.3f}, '
                  f'p95 {val["p95_latency_ms"]:.0f} ms (n={val["n"]}).\n')
report.append(f'## 4. Task 4 headline (production row)\n'
              f'faithfulness **{prod_row.faithfulness:.3f}** / answer_relevance **{prod_row.answer_relevance:.3f}** '
              f'- targets >=0.8/>=0.8: **{"MET" if T4_OK else "NOT MET"}**.\n')
Path('reports/D3/d3_ablation.md').write_text('\n'.join(report), encoding='utf-8')
print('\nwrote reports/D3/d3_ablation.md  (graded T5 deliverable)')

recommended (knee) config: {'mode': 'vector'}

wrote reports/D3/d3_ablation.md  (graded T5 deliverable)


## 9 - D4 hook: base-vs-tuned row (integrator runs this in D4)
The recommended config above is locked. In D4, Abdulrahman re-runs **just this config** with the answerer
flipped to the tuned model - no code change, pure `CSAI415_ANSWERER` swap - to get the base-vs-tuned delta.

In [12]:
# D4 ONLY - uncomment after `ollama create qwen2.5-3b-csai415` exists:
# for name in ['qwen2.5:3b-instruct', 'qwen2.5-3b-csai415']:
#     os.environ['CSAI415_ANSWERER'] = name
#     def _ans(q): return executor.answer(q, k=rec['k'], mode=rec['mode'], rerank=rec['rerank'])
#     print(name, evaluate_answers(answer_fn=_ans, gold=gold))
print('D4 hook ready - knee config:', rec)

D4 hook ready - knee config: {'mode': 'vector'}
